In [ ]:
from pathlib import Path
import uproot
import numpy as np
import matplotlib.pyplot as plt

from mutools import profit
from mutools.plotting import use_style
from mutools.plotting.helpers import mark_axis

use_style('rootlike')

INPUT = Path('/Users/mueller/Projects/spineosc/runs/legacy_root/dp_test09142026_v1_PROplot.root')
# Output configuration, both settings in one place. Every figure in this
# notebook is written to SAVE_TO with the FIG_FMT extension -- set SAVE_TO to
# None to render inline without writing anything.
SAVE_TO = Path('/Users/mueller/Projects/spineosc_technote/figures/sbnd_data_access')
FIG_FMT = 'pdf'
if SAVE_TO:
    SAVE_TO.mkdir(parents=True, exist_ok=True)


def figure_path(name):
    """Destination for one figure, or None when saving is switched off."""
    return Path(SAVE_TO) / f'{name}.{FIG_FMT}' if SAVE_TO else None

# Step 1A: area-normalised data/MC, by tier (SBND)

The $\chi^2$ PROfit draws on its channel plots is also written to the
`chi2` tree of each run's `_PROplot_histograms.root`, so the numbers below are
read back from the existing runs rather than recomputed.

**Normalisation convention.** These comparisons are **area-normalised with the
shape-only switch deliberately left off**, matching the parallel analysis.
`area-norm` rescales the prediction onto the data integral and costs one
degree of freedom, so `ndf = nbins - 1`; `shapeonly` would have gone further
and projected each systematic covariance onto per-channel shape. The `ndf`
assertion in the setup cell therefore proves area normalisation and nothing
more -- an earlier set of runs passed it while not being shape-only, which is
exactly how the two conventions got confused. `ARM` is the single switch that
selects which outputs are read, and both the tables and the panels use it.

It matters which convention is in force: the covariance here keeps its
normalisation component, so these $\chi^2$ sit below the shape-only ones.
For $\delta p$ that is 29.04 against 32.57, which lands on opposite sides of
the 0.05 threshold.

**Tiers are gated.** `ENABLED_TIERS` controls what is tabulated and plotted
anywhere in this notebook, and `DATA_OVERLAY_TIERS` separately controls which
tiers may show data points -- the request withholds data overlays for tiers 2
and 3 at this stage, so those render as MC and uncertainty band with the
$\chi^2$ and p-value quoted but no points drawn. A tier can therefore be
assessed quantitatively before any of its data is displayed.

In [ ]:
import pandas as pd
from scipy.stats import chi2 as chi2_dist

from mutools.plotting.profit import ProfitPlotData

REPO = Path('/Users/mueller/Projects/spineosc/runs/legacy_root')

# ---------------------------------------------------------------------------
# NORMALISATION CONVENTION -- one definition, used by the tables and the plots
# alike so the two can never quote different numbers.
#
# We run area-normalised and WITHOUT the shape-only switch, to match the
# parallel analysis. PROfit's `shapeonly` would additionally project every
# systematic covariance onto the per-channel shape subspace; we deliberately
# leave it off. area-norm on its own still costs one degree of freedom, so
# ndf = nbins - 1 -- that is what sbnd_chi2 asserts, and it is NOT evidence
# that a run was shape-only (an earlier set of runs passed that same assert
# while not being shape-only, which is how the two got confused before).
#
# Consequence worth remembering: the covariance entering this chi2 keeps its
# normalisation component (dp single-proton: 25.3% diagonal, against 14.1%
# under the shape projection), so these chi2 values sit a little below the
# shape-only ones. For dp that is 29.04 here versus 32.57 shape-only, which
# straddles the 0.05 threshold. Both sets of outputs exist on disk; ARM
# decides which this notebook reads.
ARM = 'cvrw_cvrw'
CHI2_LABEL = r'$\chi^2_{area}$'

# ---------------------------------------------------------------------------
# Every requested variable -> (tier, display label, channel indices).
# energy_qel is single-proton only, by construction of its XML.
VARIABLES = {
    'dp':               (1, r'$\delta p$',                     (0, 1)),
    'dpt':              (1, r'$\delta p_{T}$',                 (0, 1)),
    'dalphat':          (1, r'$\delta\alpha_{T}$',             (0, 1)),
    'dphit':            (1, r'$\delta\phi_{T}$',               (0, 1)),
    'muon_azimuth':     (1, r'$\phi_{\mu}$',                   (0, 1)),
    'proton_ke':        (1, r'$T_{p}$',                        (0, 1)),
    'muon_score':       (1, 'Muon PID score',                  (0, 1)),
    'proton_score':     (1, 'Proton PID score',                (0, 1)),
    'muon_p':           (2, r'$p_{\mu}$',                      (0, 1)),
    'muon_polar_angle': (2, r'$\theta_{\mu}$',                 (0, 1)),
    'opening_angle':    (2, r'$\theta_{\mu p}$',               (0,)),
    'vis_energy':       (3, r'$E^{reco}_{vis}$',               (0, 1)),
    'energy_qel':       (3, r'$E^{reco}_{ccqe}$',              (0,)),
}

# ---------------------------------------------------------------------------
# EMBARGO GATE. The data access request runs in stages and does not advance to
# the next tier until the previous one has been assessed, so nothing outside
# ENABLED_TIERS is tabulated, plotted or summarised anywhere below. Widen this
# tuple only once the corresponding stage has actually been cleared.
#
#   ENABLED_TIERS = (1,)        tier 1 only  (current stage)
#   ENABLED_TIERS = (1, 2)      adds tier 2
#   ENABLED_TIERS = (1, 2, 3)   adds tier 3
ENABLED_TIERS = (1,2,3)

# Tiers whose panels may carry overlaid data points. The request states that
# data will not be overlaid on tier 2 and tier 3 predictions at this stage, so
# those render as MC + uncertainty band with the chi2 and p-value quoted but
# no points drawn. Keeping this separate from ENABLED_TIERS means a tier can
# be assessed quantitatively before its data is ever shown.
DATA_OVERLAY_TIERS = (1,)

# Detector and channel indices follow the XML declaration order.
SBND = 0
CHANNELS = {0: 'ccqelike', 1: 'nproton'}
LABELS = {0: 'Single-proton', 1: 'Multi-proton'}


def tier_of(study):
    return VARIABLES[study][0]


def channels_of(study):
    """Channel indices this variable actually defines (energy_qel has one)."""
    return VARIABLES[study][2]


def studies(tiers=None):
    """(study, tier, label) for each variable in the enabled tiers."""
    tiers = ENABLED_TIERS if tiers is None else tiers
    return [(s, v[0], v[1]) for s, v in VARIABLES.items() if v[0] in tiers]


def plot_file(study, arm=ARM, tier=None):
    """Output path for a run: t<tier>_<study>_<arm>_PROplot_histograms.root."""
    tier = tier_of(study) if tier is None else tier
    return REPO / f't{tier}_{study}_{arm}_PROplot_histograms.root'


def plot_ready(path):
    """True only if the run has actually finished writing its trees.

    A job still in flight leaves a file that exists but carries no keys, so
    Path.exists() is not enough to decide a run is usable.
    """
    if not path.exists():
        return False
    try:
        with uproot.open(path) as handle:
            keys = {k.split(';')[0] for k in handle.keys(recursive=False)}
        return {'hist1d', 'chi2'} <= keys
    except Exception:
        return False


def sbnd_chi2(path, channel):
    """chi2, nbins, ndf and p-value for one SBND channel of one run.

    The assertion checks AREA NORMALISATION, which costs the one degree of
    freedom. It says nothing about whether `shapeonly` was set -- that is
    read off the ini, and ARM above records which convention we are on.
    """
    value, nbins, ndf = ProfitPlotData(path).get_chi2(variable=0, detector=SBND, channel=channel)
    assert ndf == nbins - 1, f'{path.name}: ndf {ndf} != nbins-1 ({nbins - 1}); not area-normalised?'
    return value, nbins, ndf, float(chi2_dist.sf(value, ndf))

Several studies have more than one run on disk. The table names the file each
number came from, so a stale or rebinned run is visible rather than silently
picked up -- `dp`, for instance, has both a 20-bin and a newer 10-bin run.

In [ ]:
rows, missing = [], []
for study, tier, label in studies():
    path = plot_file(study)
    if not plot_ready(path):
        missing.append(f'{study} (tier {tier})')
        continue
    for channel in channels_of(study):
        value, nbins, ndf, p = sbnd_chi2(path, channel)
        rows.append({'tier': tier, 'variable': label, 'channel': CHANNELS[channel],
                     'file': path.name, 'chi2': value, 'ndf': ndf,
                     'chi2/ndf': value / ndf, 'p-value': p})

def show(frame):
    fmt = lambda v: f'{v:.2e}' if 0 < abs(v) < 1e-3 else f'{v:.3f}'
    with pd.option_context('display.width', 200, 'display.max_colwidth', 52):
        print(frame.to_string(index=False, float_format=fmt))

results = pd.DataFrame(rows)
print(f'Convention: ARM = {ARM!r}  (area-normalised, shape-only NOT set)')
print(f'Tiers shown: {ENABLED_TIERS}   data overlaid on tiers: {DATA_OVERLAY_TIERS}\n')
show(results)

below = results[results['p-value'] < 0.05]
print(f'\n{len(below)} of {len(results)} comparisons below the 0.05 threshold')
for _, r in below.iterrows():
    print(f"  {r['variable']} / {r['channel']}: p = {r['p-value']:.4f}")
if missing:
    print('\nnot on disk yet: ' + ', '.join(missing))

## Per-variable p-values

The document's off-ramp is any p-value below 0.05, so the threshold is drawn
rather than left to the reader.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
shown = results.dropna(subset=['p-value'])
labels = list(dict.fromkeys(shown['variable']))
x = np.arange(len(labels))

entries = [
    ('Single-proton', 'C1', 'ccqelike'),
    ('Multi-proton', 'C2', 'nproton')
]

for channel_name, color, channel_key in entries:
    subset = shown[shown['channel'] == channel_key].set_index('variable').reindex(labels)
    ax.scatter(x, subset['p-value'], s=45, color=color, label=channel_name, zorder=3)
    
ax.axhline(0.05, color='#eb6834', lw=1.2, ls='--', zorder=2, label='p = 0.05')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Shape-only p-value')
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylim(min(shown['p-value'].min() / 5, 1e-2), 2.0)
ax.legend(loc='lower right')
mark_axis(ax, rf'$\bf{{SBND}}$ Data Access', vadj=0.05)
mark_axis(ax, rf'$4.51 \times 10^{{18}}$ POT', vadj=0.05, alignment='right')

if SAVE_TO:
    fig.savefig(figure_path(f'pvalues_{ARM}'), bbox_inches='tight')
plt.show()

## Caveats carried by these numbers

- They come from whatever detector-systematics set and selection the source
  runs used; the filenames in the table date them.
- The dirt samples are empty in the inputs (`events/sbnd_dirt/selected` has
  zero entries), so the prediction has no dirt component.
- The $\chi^2$ convention is whichever `--chi2` the run used (PROfit's default
  is Neyman, stat covariance from data).
- The prediction is rescaled to the data integral per channel, so the data POT
  mismatch in the `_data.xml` files does not affect these numbers. It will
  matter for the shape+normalisation stage, which turns area normalisation off
  and compares absolute rates.
- The systematics are *not* shape-projected, so a systematic that is mostly an
  overall scale still contributes to the covariance even though the
  normalisation it would move has already been divided out.

# Step 1B: data/MC panels

One single-variable config per observable, read at whichever `ARM` the setup
cell selects. Every panel is **area normalized**, so the legend quotes each
subchannel as a percentage of the prediction and no absolute rate is shown.

Which variables appear is controlled by `ENABLED_TIERS`; whether their data
points are drawn is controlled separately by `DATA_OVERLAY_TIERS`. Panels for
a tier outside the latter are rendered as MC plus uncertainty band, with the
$\chi^2$ and p-value still quoted in the legend.

Runs still in progress simply show as missing, so these cells are safe to
re-run while a batch is going.

In [ ]:
import shutil
import matplotlib.pyplot as plt
from mutools.plotting import run
import mutools.plotting as mp

# FIG_FMT is defined in the setup cell alongside SAVE_TO. The render cell globs
# for the staged file and names the final file with this same extension -- they
# used to disagree, which silently produced PDFs called .png.
mp.saver.configure(fmt=FIG_FMT, dpi=300, rasterized=True, ytick_precision=2, mark_hadj=0.055)

DETECTORS = {0: 'SBND',}# 1: 'ICARUS Run 2', 2: 'ICARUS Run 4'}

# ARM, VARIABLES, ENABLED_TIERS and DATA_OVERLAY_TIERS all live in the setup
# cell above, so the tables and the plots cannot drift onto different runs.

# CHANNELS (setup cell) holds the XML channel names, which are what the
# selection actually is. These two maps are presentation only:
#   CHANNEL_LABELS -> what the reader sees on the plot and in the tables
#   CHANNEL_TOKENS -> the filename fragment, matching the technote's
#                     datamc_<detector>_<variable>_<singlep|multip>.pdf
CHANNEL_LABELS = {0: 'Single-Proton', 1: 'Multi-Proton'}
CHANNEL_TOKENS = {0: 'singlep', 1: 'multip'}

# Labels and colours use OPPOSITE orders, which is easy to get wrong:
#   - SUBCHANNELS is the XML declaration order REVERSED, because mutools pairs
#     the first label with the last subchannel.
#   - SUBCHANNEL_COLORS is the XML declaration order AS WRITTEN
#     (intime, cos, nc, nuecc, othercc, target).
# Verified against a rendered panel: Target comes out #99CCFF, not #FBBC04.
SUBCHANNELS = ['Target', r'Other $\nu$ CC', r'$\nu_e$ CC', r'$\nu$ NC',
               'Cosmics', 'In-time Cosmics']
SUBCHANNEL_COLORS = ['C0', 'C1', 'C2', 'C4', 'C3', 'C5']

# study -> (x-axis label, xlim). Ranges match the var0 binning in the XMLs.
# xlim may be a single pair, or a {channel: pair} dict where the two channels
# are binned over different ranges (vis_energy).
PLOT_AXES = {
    'dp':               (r'Reconstructed $\delta p$ [MeV/c]',       [0.0, 1000.0]),
    'dpt':              (r'Reconstructed $\delta p_{T}$ [MeV/c]',   [0.0, 1000.0]),
    'dalphat':          (r'Reconstructed $\delta\alpha_{T}$ [rad]', [0.0, 3.14]),
    'dphit':            (r'Reconstructed $\delta\phi_{T}$ [rad]',   [0.0, 3.14]),
    'muon_score':       ('Muon PID Score',                          [0.30, 1.00]),
    'proton_score':     ('Proton PID Score',                        [0.40, 1.00]),
    'muon_azimuth':     ('Muon Azimuthal Angle [rad]',              [-3.14, 3.14]),
    'proton_ke':        ('Leading Proton Kinetic Energy [MeV]',     [40.0, 600.0]),
    'muon_p':           ('Muon Momentum [MeV/c]',                   [225.0, 1225.0]),
    'muon_polar_angle': ('Muon Polar Angle [rad]',                  [0.0, 3.14]),
    'opening_angle':    ('Muon-Proton Opening Angle [rad]',         [0.0, 3.14]),
    'vis_energy':       ('Reconstructed Visible Energy [GeV]',      {0: [0.3, 1.5], 1: [0.4, 2.0]}),
    'energy_qel':       (r'Reconstructed $E_{ccqe}$ [GeV]',         [0.2, 1.5]),
}

# Per-variable y-extent, one entry per variable so a variable is comparable
# across detectors and channels. Each value is ~55% above the tallest of
# (CV, band top, and for tier 1 also data + error); the report cell below
# prints the measured peaks. Tier 2 and 3 entries were set from CV and band
# only, so that setting a y-scale never required looking at their data.
PLOT_YLIM = {
    'dp':               [0.0, 0.006],
    'dpt':              [0.0, 0.006],
    'dalphat':          [0.0, 1.5],
    'dphit':            [0.0, 3.0],
    'muon_score':       [0.0, 15.0],
    'proton_score':     [0.0, 50.0],
    'muon_azimuth':     [0.0, 0.50],
    'proton_ke':        [0.0, 0.01],
    'muon_p':           [0.0, 0.005],
    'muon_polar_angle': [0.0, 1.11],
    'opening_angle':    [0.0, 1.44],
    'vis_energy':       [0.0, 3.18],
    'energy_qel':       [0.0, 2.44],
}
# (variable,), (variable, detector) or (variable, detector, channel) -- most specific wins.
PLOT_YLIM_OVERRIDES = {}

PLOT_LEGEND = {}                 # variable -> (loc, ncol)
DEFAULT_LEGEND = ('upper right', 2)

CODE_VERSION = 'v3.0.4'
SELECTION_VERSION = 'v1.1.0'


def plot_axes(study, channel):
    """(xlabel, xlim) for one panel, resolving per-channel xlim if present."""
    xlabel, xlim = PLOT_AXES[study]
    if isinstance(xlim, dict):
        xlim = xlim.get(channel)
    return xlabel, xlim


def plot_ylim(variable, detector, channel):
    for key in ((variable, detector, channel), (variable, detector), (variable,)):
        if key in PLOT_YLIM_OVERRIDES:
            return PLOT_YLIM_OVERRIDES[key]
    return PLOT_YLIM.get(variable)

What is on disk and how tall each panel is, so `PLOT_YLIM` can be set to
real numbers instead of autoscale.

In [ ]:
from mutools.plotting.profit import TraceType

# Peak heights, so PLOT_YLIM can be set to real numbers instead of autoscale.
# For tiers outside DATA_OVERLAY_TIERS the data trace is deliberately NOT read,
# so a y-scale can be chosen without looking at withheld data; those rows are
# marked "CV+band" and may clip if the data later turns out to sit higher.
print(f"{'study':24s}{'tier':>5}{'det':>4}{'ch':>4}{'peak':>10}{'basis':>14}{'ylim in use':>18}")
for study, tier, _label in studies():
    path = plot_file(study)
    if not plot_ready(path):
        state = 'IN PROGRESS' if path.exists() else 'MISSING'
        print(f'{study:24s}{tier:5d}{state:>36}')
        continue
    d = ProfitPlotData(path, True)
    use_data = tier in DATA_OVERLAY_TIERS
    for det in DETECTORS:
        for ch in channels_of(study):
            key = f'0:0:{det}:{ch}:total'
            try:
                cv = d.get_trace(f'{key}:CV', TraceType.HIST_CONTENTS)
                band = d.get_trace(f'{key}:CV', TraceType.HIST_ERROR_BAND)
                stack = [cv[:, 3], band[:, 1] + band[:, 3]]
                if use_data:
                    dat = d.get_trace(f'{key}:DATA', TraceType.HIST_CONTENTS)
                    derr = dat[:, 4] if dat.shape[1] > 4 else np.sqrt(np.maximum(dat[:, 3], 0.0))
                    stack.append(dat[:, 3] + derr)
            except Exception as exc:
                print(f'{study:24s}{tier:5d}{det:4d}{ch:4d}   trace error: {exc}')
                continue
            peak = float(np.nanmax(np.concatenate(stack)))
            basis = 'CV+band+data' if use_data else 'CV+band'
            print(f'{study:24s}{tier:5d}{det:4d}{ch:4d}{peak:10.4f}{basis:>14}'
                  f'{str(plot_ylim(study, det, ch)):>18}')

Render. Missing runs are skipped, so this can be re-run while the batch is
still going. Filenames follow
`sbndaccess_<variable>_<detector>_<channel>_<arm>.<FIG_FMT>`, where `FIG_FMT`
is the format set in the setup cell.

In [ ]:
made, skipped, suppressed = 0, [], []
for study, tier, _label in studies():
    path = plot_file(study)
    if not plot_ready(path):
        skipped.append(f'{study}/tier{tier}' + ('' if not path.exists() else ' (in progress)'))
        continue
    legend_loc, legend_ncol = PLOT_LEGEND.get(study, DEFAULT_LEGEND)

    # The request withholds data overlays for tiers outside DATA_OVERLAY_TIERS.
    # ratio='data' is the ONLY thing that draws data points in mutools, so
    # ratio=None gives MC + band with no points and no ratio panel; the chi2
    # and p-value still come from the file's chi2 tree either way.
    overlay = tier in DATA_OVERLAY_TIERS
    if not overlay:
        suppressed.append(f'{study} (tier {tier})')

    for det, det_label in DETECTORS.items():
        for ch in channels_of(study):
            xlabel, xlim = plot_axes(study, ch)
            run({
                'general': {
                    'input': str(path),
                    'output': str(SAVE_TO) if SAVE_TO else '',
                    'savefig': SAVE_TO is not None,
                    'code_version': CODE_VERSION,
                    'selection_version': SELECTION_VERSION,
                    'subchannels': SUBCHANNELS,
                    # Full detector list for the same reason as channels:
                    # dispatch indexes it with the absolute detector index.
                    'detectors': list(DETECTORS.values()),
                    # Full channel list: dispatch indexes it with the
                    # absolute 'channel' below, so a 1-element list breaks ch=1.
                    'channels': [CHANNEL_LABELS[i] for i in sorted(CHANNEL_LABELS)],
                    'scale_by_width': 'enabled',
                    'counter_index': 4,      # Count is var4 in the coverage convention
                },
                'plot': [{
                    'type': 'histogram',
                    'variable': 0,
                    'channel': ch,
                    'detectors': [det],
                    'colors': SUBCHANNEL_COLORS,
                    'xlabel': xlabel,
                    'ylabel': 'Area Normalized',
                    'xlim': xlim,
                    'ylim': plot_ylim(study, det, ch),
                    'rlim': [0.0, 2.0],
                    'ratio': 'data' if overlay else None,
                    'scale_by_width': 'enabled',
                    'disable_systematics': False,
                    'band_label': 'Total Uncertainty',
                    # Percentages, never yields.
                    'counter_fmt': '.1%',
                    'watermark': r'$\bf{SBN}$ Analysis-in-Progress',
                    'show_chi2': True,
                    'chi2_label': CHI2_LABEL,
                    'show_pvalue': True,
                    'legend_ncol': legend_ncol,
                    'legend_loc': legend_loc,
                }],
            })
            if SAVE_TO:
                # mutools picks the staged filename itself (hist_<i>_<j>_<version>.<fmt>,
                # and the indices do not map cleanly onto detector/channel), so take
                # whichever staged file it just wrote rather than predicting the name.
                staged = sorted(Path(SAVE_TO).glob(f'hist_*_{CODE_VERSION}.{FIG_FMT}'),
                                key=lambda q: q.stat().st_mtime)
                if not staged:
                    raise FileNotFoundError(f'{study} det{det} ch{ch}: '
                                            f'no staged .{FIG_FMT} appeared')
                target = figure_path(
                    f'sbndaccess_{study}_{det_label.replace(" ", "").lower()}'
                    f'_{CHANNEL_TOKENS[ch]}_{ARM}')
                shutil.move(staged[-1], target)
            else:
                plt.show()
            made += 1

print(f'{made} panels rendered for tiers {ENABLED_TIERS}')
if suppressed:
    print('data points withheld (not in DATA_OVERLAY_TIERS): ' + ', '.join(suppressed))
if skipped:
    print('not produced yet: ' + ', '.join(skipped))

## Tier 1 summary

Tier 1 runs the shifted arm only, so there is no vanilla comparison. This is
the headline table: chi2, ndf and p-value for each variable in each channel.

Two things to keep attached to these numbers. chi2 carries roughly +/-0.6 of
run-to-run scatter from the unpinned seed in the spline-to-covariance throws,
and with 16 comparisons a single p-value near 0.01 is unremarkable on its own.


In [ ]:
# Summary: chi2, ndf and p-value per variable and channel, for the enabled
# tiers only. Reads the same ARM as the plots, so the panel and this table
# always quote the same number.
rows = []
for study, tier, label in studies():
    path = plot_file(study)
    if not plot_ready(path):
        print(f'missing: {study} (tier {tier})')
        continue
    for ch in channels_of(study):
        value, nbins, ndf, p = sbnd_chi2(path, ch)
        rows.append({'tier': tier, 'variable': label, 'channel': CHANNEL_LABELS[ch],
                     'chi2': value, 'ndf': ndf, 'chi2/ndf': value / ndf, 'p-value': p})
summary = pd.DataFrame(rows)
show(summary)
low = summary[summary['p-value'] < 0.05]
print(f"\n{len(low)} of {len(summary)} comparisons below p = 0.05")
# by name, not itertuples position: 'p-value' is renamed by the hyphen and
# its index shifts if a column is ever added
for _, r in low.iterrows():
    print(f"  {r['variable']} / {r['channel']}: p = {r['p-value']:.4f}")

# Step 2: shape + normalisation (phase 2)

Everything above is Step 1: area-normalised, so the prediction is rescaled onto
the data integral per channel and only the shape is being compared. Step 2 drops
that rescaling and compares **absolute rates**.

**Two consequences that are easy to get wrong.** Without area normalisation the
normalisation constraint is gone, so `ndf = nbins`, not `nbins - 1` -- the
phase-1 helper asserts the latter and would fail here, which is why this section
has its own accessor. And the POT matching between the MC and `_data.xml`
configs now matters directly, where in Step 1 it divided out.

**The gating below is deliberately separate from phase 1's.** `PHASE2_*` does
not read `ENABLED_TIERS`, so widening phase 1 to tier 2 or 3 does not drag phase
2 along with it. Each phase is assessed and cleared on its own, in its own
staged order, exactly as the request describes.

These read the same `t<N>_<var>_cvrw` caches as phase 1 -- neither `shapeonly`
nor `area-norm` is part of the cache hash -- so the runs are seconds, not
minutes. Configs are `<var>_cvrw_phase2_plot.ini`.

In [ ]:
# ---------------------------------------------------------------------------
# PHASE 2 CONFIGURATION. Independent of the phase-1 switches above by design:
# nothing here reads ENABLED_TIERS or DATA_OVERLAY_TIERS, so moving phase 1 on
# to a later tier leaves phase 2 where it is.
PHASE2_ARM = 'cvrw_cvrw_phase2'
PHASE2_CHI2_LABEL = r'$\chi^2_{abs}$'

PHASE2_ENABLED_TIERS = (1,2,3)
# None (or an empty tuple) means no tier shows data points: phase 2 compares
# absolute rates, so an overlay puts the normalisation on screen. The helpers
# below coerce it, so None and () behave identically and neither raises.
PHASE2_DATA_OVERLAY_TIERS = None

# Absolute rates span a different range per variable than the area-normalised
# shapes, so the phase-1 PLOT_YLIM values do not transfer. The real table is
# hardcoded further down; this fallback only keeps the render cell importable
# if that one has not been run yet.
PHASE2_YLIM = {}


def phase2_file(study, tier=None):
    """Output path for a phase-2 run."""
    tier = tier_of(study) if tier is None else tier
    return REPO / f't{tier}_{study}_{PHASE2_ARM}_PROplot_histograms.root'


def phase2_chi2(path, channel):
    """chi2, nbins, ndf and p-value for one SBND channel of a phase-2 run.

    The assertion is the MIRROR of the phase-1 one: with no area normalisation
    there is no normalisation constraint, so ndf must equal nbins. A file that
    fails this is an area-normalised run that has been mislabelled, which is
    exactly the confusion this notebook has already been through once.
    """
    value, nbins, ndf = ProfitPlotData(path).get_chi2(variable=0, detector=SBND, channel=channel)
    assert ndf == nbins, f'{path.name}: ndf {ndf} != nbins {nbins}; is this really not area-normalised?'
    return value, nbins, ndf, float(chi2_dist.sf(value, ndf))


def phase2_overlay_tiers():
    """Overlay tiers as a tuple, treating None as 'none'."""
    return tuple(PHASE2_DATA_OVERLAY_TIERS or ())


def phase2_studies():
    """(study, tier, label) for the phase-2 enabled tiers."""
    return [(s, v[0], v[1]) for s, v in VARIABLES.items() if v[0] in tuple(PHASE2_ENABLED_TIERS or ())]


# A staged analysis clears Step 1 for a tier before Step 2 touches it. This is a
# warning rather than an assertion: the two are meant to be independent, and
# there are legitimate reasons to look at phase 2 for a tier out of order, but
# doing it silently is not one of them.
_ahead = set(PHASE2_ENABLED_TIERS or ()) - set(ENABLED_TIERS or ())
if _ahead:
    print(f'NOTE: phase 2 is enabled for tier(s) {sorted(_ahead)} that phase 1 has not cleared '
          f'(phase 1 is at {ENABLED_TIERS}).')
print(f'phase 2: ARM={PHASE2_ARM!r}  tiers={PHASE2_ENABLED_TIERS}  '
      f'data overlaid on={phase2_overlay_tiers() or "nothing"}')
print(f'phase 1: ARM={ARM!r}  tiers={ENABLED_TIERS}  data overlaid on={DATA_OVERLAY_TIERS}')

Absolute-rate $\chi^2$ per variable and channel. `ndf = nbins` here,
so these are not comparable bin-for-bin with the Step 1 table above.

In [ ]:
rows, missing = [], []
for study, tier, label in phase2_studies():
    path = phase2_file(study)
    if not plot_ready(path):
        missing.append(f'{study} (tier {tier})')
        continue
    for channel in channels_of(study):
        value, nbins, ndf, p = phase2_chi2(path, channel)
        rows.append({'tier': tier, 'variable': label, 'channel': CHANNELS[channel],
                     'chi2': value, 'nbins': nbins, 'ndf': ndf,
                     'chi2/ndf': value / ndf, 'p-value': p})

phase2 = pd.DataFrame(rows)
if len(phase2):
    show(phase2)
    low = phase2[phase2['p-value'] < 0.05]
    print(f"\n{len(low)} of {len(phase2)} below p = 0.05")
    for _, r in low.iterrows():
        print(f"  {r['variable']} / {r['channel']}: p = {r['p-value']:.4f}")
else:
    print('no phase-2 outputs on disk yet')
if missing:
    print('\nnot produced yet: ' + ', '.join(missing))
    print('  run:  ../profit-asymmetric/build/bin/PROfit '
          '--config xml/sbnd_data_access/phase1-tier<N>/<var>_cvrw_phase2_plot.ini plot')

## Per-variable p-values, absolute normalisation

The phase-1 panel's counterpart. Same 0.05 threshold and log scale, so the two
can be read side by side -- but these carry the normalisation, so a variable can
sit lower here than in phase 1 without its shape having changed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
shown = phase2.dropna(subset=['p-value'])
labels = list(dict.fromkeys(shown['variable']))
x = np.arange(len(labels))

entries = [
    ('Single-proton', 'C1', 'ccqelike'),
    ('Multi-proton', 'C2', 'nproton')
]

for channel_name, color, channel_key in entries:
    subset = shown[shown['channel'] == channel_key].set_index('variable').reindex(labels)
    ax.scatter(x, subset['p-value'], s=45, color=color, label=channel_name, zorder=3)

ax.axhline(0.05, color='#eb6834', lw=1.2, ls='--', zorder=2, label='p = 0.05')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Absolutely-normalized p-value')
ax.set_xlim(-0.5, len(labels) - 0.5)
ax.set_ylim(min(shown['p-value'].min() / 5, 1e-2), 2.0)
ax.legend(loc='lower right')
mark_axis(ax, rf'$\bf{{SBND}}$ Data Access', vadj=0.05)
mark_axis(ax, rf'$4.51 \times 10^{{18}}$ POT', vadj=0.05, alignment='right')

if SAVE_TO:
    fig.savefig(figure_path(f'pvalues_{PHASE2_ARM}'), bbox_inches='tight')
plt.show()

Peak heights for `PHASE2_YLIM`. As in phase 1, the data trace is read
only for tiers in `PHASE2_DATA_OVERLAY_TIERS`, so a y-scale can be chosen for a
withheld tier without looking at its data.

In [ ]:
# Phase-2 y-limits, hardcoded PER CHANNEL.
#
# Absolute normalisation makes the channels differ in scale: the single-proton
# peak runs 2.8 to 6.2 times the multi-proton one across these variables
# (median 3.4), so one shared limit per variable would flatten every
# multi-proton panel by about that factor. Phase 1 could share a limit because
# area normalisation put both channels on the same footing; here it cannot.
#
# Measured from CV + band only, since phase-2 overlays are off. Each value is
# that peak plus ~55% headroom, rounded UP to three significant figures; the
# peaks are recorded beneath each entry.
PHASE2_YLIM = {
    'dp':                  {0: [0.0, 20.0], 1: [0.0, 4.00]},
    'dpt':                 {0: [0.0, 22.0], 1: [0.0, 5.00]},
    'dalphat':             {0: [0.0, 3800], 1: [0.0, 1200]},
    'dphit':               {0: [0.0, 10000], 1: [0.0, 1600]},
    'muon_score':          {0: [0.0, 45000], 1: [0.0, 13000]},
    'proton_score':        {0: [0.0, 160000], 1: [0.0, 53000]},
    'muon_azimuth':        {0: [0.0, 1400], 1: [0.0, 500]},
    'proton_ke':           {0: [0.0, 35.0], 1: [0.0, 9.75]},
    'proton_p':            {0: [0.0, 15.0], 1: [0.0, 4.97]},
    'muon_p':              {0: [0.0, 17.0], 1: [0.0, 5.81]},
    'muon_polar_angle':    {0: [0.0, 4500], 1: [0.0, 1600]},
    'opening_angle':       {0: [0.0, 5500], 1: [0.0, 1400]},
    'muon_ke':             {0: [0.0, 20.0], 1: [0.0, 7.00]},
    'vis_energy':          {0: [0.0, 12500], 1: [0.0, 4000]},
    'energy_qel':          {0: [0.0, 9500]},
}


def phase2_ylim(study, channel):
    """y-limit for one phase-2 panel, or None to autoscale.

    Falls back to None rather than to another channel's limit: borrowing the
    single-proton scale for a multi-proton panel is the exact mistake this
    table exists to prevent, and an autoscaled panel is obvious where a
    silently squashed one is not.
    """
    return PHASE2_YLIM.get(study, {}).get(channel)


print(f'PHASE2_YLIM: {len(PHASE2_YLIM)} variables, '
      f'{sum(len(v) for v in PHASE2_YLIM.values())} channel entries')

Phase-2 panels. The y-axis is a rate, not a normalised shape, so the
legend's subchannel percentages still read as fractions of the prediction but the
scale is absolute. Data points follow `PHASE2_DATA_OVERLAY_TIERS`.

In [ ]:
made, skipped, suppressed = 0, [], []
for study, tier, _label in phase2_studies():
    path = phase2_file(study)
    if not plot_ready(path):
        skipped.append(f'{study}/tier{tier}')
        continue
    legend_loc, legend_ncol = PLOT_LEGEND.get(study, DEFAULT_LEGEND)
    overlay = tier in phase2_overlay_tiers()
    if not overlay:
        suppressed.append(f'{study} (tier {tier})')
    for det, det_label in DETECTORS.items():
        for ch in channels_of(study):
            xlabel, xlim = plot_axes(study, ch)
            run({
                'general': {
                    'input': str(path),
                    'output': str(SAVE_TO) if SAVE_TO else '',
                    'savefig': SAVE_TO is not None,
                    'code_version': CODE_VERSION,
                    'selection_version': SELECTION_VERSION,
                    'subchannels': SUBCHANNELS,
                    'detectors': list(DETECTORS.values()),
                    'channels': [CHANNEL_LABELS[i] for i in sorted(CHANNEL_LABELS)],
                    'scale_by_width': 'enabled',
                    'counter_index': 4,
                },
                'plot': [{
                    'type': 'histogram',
                    'variable': 0,
                    'channel': ch,
                    'detectors': [det],
                    'colors': SUBCHANNEL_COLORS,
                    'xlabel': xlabel,
                    'ylabel': 'Events / bin width',
                    'xlim': xlim,
                    'ylim': phase2_ylim(study, ch),
                    'rlim': [0.0, 2.0],
                    'ratio': 'data' if overlay else None,
                    'scale_by_width': 'enabled',
                    'disable_systematics': False,
                    'band_label': 'Total Uncertainty',
                    'counter_fmt': '.1%',
                    'watermark': r'$\bf{SBN}$ Analysis-in-Progress',
                    'show_chi2': True,
                    'chi2_label': PHASE2_CHI2_LABEL,
                    'show_pvalue': True,
                    'legend_ncol': legend_ncol,
                    'legend_loc': legend_loc,
                }],
            })
            if SAVE_TO:
                staged = sorted(Path(SAVE_TO).glob(f'hist_*_{CODE_VERSION}.{FIG_FMT}'),
                                key=lambda q: q.stat().st_mtime)
                if not staged:
                    raise FileNotFoundError(f'{study} det{det} ch{ch}: no staged .{FIG_FMT} appeared')
                target = figure_path(
                    f'sbndaccess_{study}_{det_label.replace(" ", "").lower()}'
                    f'_{CHANNEL_TOKENS[ch]}_{PHASE2_ARM}')
                shutil.move(staged[-1], target)
            else:
                plt.show()
            made += 1

print(f'{made} phase-2 panels rendered for tiers {PHASE2_ENABLED_TIERS}')
if suppressed:
    print('data points withheld: ' + ', '.join(suppressed))
if skipped:
    print('not produced yet: ' + ', '.join(skipped))

# Step 3: visual inspection, absolute normalisation

Steps 1 and 2 reduce each comparison to a number. This step looks at the
panels, with data overlaid on an absolutely normalised prediction, for the
qualitative features a p-value will not show: a slope the band happens to
cover, a single discrepant bin, structure at a threshold, a tail that drifts.

It reads the **same files as phase 2** -- no new runs -- and differs only in
drawing the data. The tier switches are separate again, so enabling a tier here
does not touch phase 1 or phase 2 and vice versa.

Because the prediction is absolutely normalised, the overlay puts the
normalisation on screen. `PHASE3_ENABLED_TIERS` is the gate for that.

In [ ]:
# ---------------------------------------------------------------------------
# PHASE 3 CONFIGURATION. Independent of both phases above: nothing here reads
# ENABLED_TIERS or PHASE2_ENABLED_TIERS, so the three advance separately.
PHASE3_ARM = PHASE2_ARM          # same outputs as phase 2; only the drawing differs
PHASE3_CHI2_LABEL = PHASE2_CHI2_LABEL

PHASE3_ENABLED_TIERS = (1,2,3)
# Phase 3 IS the overlay step, so this normally tracks PHASE3_ENABLED_TIERS.
# Kept separate so a tier can be rendered band-only first if wanted. None or ()
# means no tier draws data.
PHASE3_DATA_OVERLAY_TIERS = (1,2,3)

# Phase-3 y-limits, hardcoded PER CHANNEL and independent of phase 2.
#
# They are NOT the phase-2 numbers reused. Phase 2 draws no ratio panel, so its
# main axis is the whole figure height; phase 3 does, and mutools builds the
# pair with height_ratios=[4, 1] (mutools/plotting/profit.py), leaving the main
# axis 4/5 as tall. The legend is sized in points rather than axis fractions, so
# it covers 1/0.8 as much of a shorter axis and the phase-2 values leave visibly
# less clearance once the ratio panel appears.
#
# Each value below started as the phase-2 limit x 1.159 -- the factor that holds
# the ABSOLUTE clearance above the peak fixed across the two geometries -- and
# was then rounded UP to a round number. The arithmetic is recorded beneath each
# entry. Tune freely from here: nothing recomputes them.
PHASE3_YLIM = {
    'dp':                  {0: [0.0, 25], 1: [0.0, 7]},
    'dpt':                 {0: [0.0, 30], 1: [0.0, 6]},
    'dalphat':             {0: [0.0, 4500], 1: [0.0, 1500]},
    'dphit':               {0: [0.0, 12000], 1: [0.0, 2000]},
    'muon_score':          {0: [0.0, 60000], 1: [0.0, 18000]},
    'proton_score':        {0: [0.0, 200000], 1: [0.0, 70000]},
    'muon_azimuth':        {0: [0.0, 1800], 1: [0.0, 600]},
    'proton_ke':           {0: [0.0, 45], 1: [0.0, 12]},
    'proton_p':            {0: [0.0, 18], 1: [0.0, 6]},
    'muon_p':              {0: [0.0, 20], 1: [0.0, 7]},
    'muon_polar_angle':    {0: [0.0, 6000], 1: [0.0, 2000]},
    'opening_angle':       {0: [0.0, 8000], 1: [0.0, 2000]},
    'muon_ke':             {0: [0.0, 25], 1: [0.0, 9]},
    'vis_energy':          {0: [0.0, 18000], 1: [0.0, 5000]},
    'energy_qel':          {0: [0.0, 14000]},
}


def phase3_ylim(study, channel):
    """y-limit for one phase-3 panel, or None to autoscale."""
    return PHASE3_YLIM.get(study, {}).get(channel)


def phase3_file(study, tier=None):
    tier = tier_of(study) if tier is None else tier
    return REPO / f't{tier}_{study}_{PHASE3_ARM}_PROplot_histograms.root'


def phase3_overlay_tiers():
    return tuple(PHASE3_DATA_OVERLAY_TIERS or ())


def phase3_studies():
    return [(s, v[0], v[1]) for s, v in VARIABLES.items()
            if v[0] in tuple(PHASE3_ENABLED_TIERS or ())]


for _name, _tiers in (('phase 1', ENABLED_TIERS), ('phase 2', PHASE2_ENABLED_TIERS)):
    _ahead = set(PHASE3_ENABLED_TIERS or ()) - set(_tiers or ())
    if _ahead:
        print(f'NOTE: phase 3 is enabled for tier(s) {sorted(_ahead)} that {_name} has not cleared.')
print(f'phase 3: ARM={PHASE3_ARM!r}  tiers={PHASE3_ENABLED_TIERS}  '
      f'data overlaid on={phase3_overlay_tiers() or "nothing"}')
print(f'         y-limits: {sum(len(v) for v in PHASE3_YLIM.values())} hardcoded channel entries')

Does any data point sit above its configured y-limit? Run this before
rendering: a clipped point is easy to miss in a finished panel and reads as a
missing marker rather than a disagreement.

In [ ]:
clipped = []
for study, tier, _label in phase3_studies():
    path = phase3_file(study)
    if not plot_ready(path):
        print(f'{study}: MISSING'); continue
    if tier not in phase3_overlay_tiers():
        continue                      # no data drawn, nothing to clip
    d = ProfitPlotData(path, True)
    for ch in channels_of(study):
        k = f'0:0:0:{ch}:total'
        dat = d.get_trace(f'{k}:DATA', TraceType.HIST_CONTENTS)
        derr = dat[:, 4] if dat.shape[1] > 4 else np.sqrt(np.maximum(dat[:, 3], 0.0))
        top = float(np.nanmax(dat[:, 3] + derr))
        lim = phase3_ylim(study, ch)
        if lim is not None and top > lim[1]:
            clipped.append((study, ch, top, lim[1]))
if clipped:
    print('Data exceeds the limit -- raise these PHASE3_YLIM entries:')
    for study, ch, top, lim in clipped:
        print(f'  {study:22s} ch{ch}  data top {top:.4g} > ylim {lim:.4g}  '
              f'(suggest {1.1 * top:.4g})')
else:
    print(f'no clipping: every drawn data point fits inside its y-limit '
          f'({len(phase3_studies())} variables, tiers {PHASE3_ENABLED_TIERS})')

Render. Panels are written with the `phase3` suffix so they do not
overwrite the phase-2 band-only versions of the same variable.

In [ ]:
made, skipped, suppressed = 0, [], []
for study, tier, _label in phase3_studies():
    path = phase3_file(study)
    if not plot_ready(path):
        skipped.append(f'{study}/tier{tier}')
        continue
    legend_loc, legend_ncol = PLOT_LEGEND.get(study, DEFAULT_LEGEND)
    overlay = tier in phase3_overlay_tiers()
    if not overlay:
        suppressed.append(f'{study} (tier {tier})')
    for det, det_label in DETECTORS.items():
        for ch in channels_of(study):
            xlabel, xlim = plot_axes(study, ch)
            run({
                'general': {
                    'input': str(path),
                    'output': str(SAVE_TO) if SAVE_TO else '',
                    'savefig': SAVE_TO is not None,
                    'code_version': CODE_VERSION,
                    'selection_version': SELECTION_VERSION,
                    'subchannels': SUBCHANNELS,
                    'detectors': list(DETECTORS.values()),
                    'channels': [CHANNEL_LABELS[i] for i in sorted(CHANNEL_LABELS)],
                    'scale_by_width': 'enabled',
                    'counter_index': 4,
                },
                'plot': [{
                    'type': 'histogram',
                    'variable': 0,
                    'channel': ch,
                    'detectors': [det],
                    'colors': SUBCHANNEL_COLORS,
                    'xlabel': xlabel,
                    'ylabel': 'Events / bin width',
                    'xlim': xlim,
                    'ylim': phase3_ylim(study, ch),
                    'rlim': [0.0, 2.0],
                    'ratio': 'data' if overlay else None,
                    'scale_by_width': 'enabled',
                    'disable_systematics': False,
                    'band_label': 'Total Uncertainty',
                    'counter_fmt': '.1%',
                    'watermark': r'$\bf{SBN}$ Analysis-in-Progress',
                    'show_chi2': True,
                    'chi2_label': PHASE3_CHI2_LABEL,
                    'show_pvalue': True,
                    'legend_ncol': legend_ncol,
                    'legend_loc': legend_loc,
                }],
            })
            if SAVE_TO:
                staged = sorted(Path(SAVE_TO).glob(f'hist_*_{CODE_VERSION}.{FIG_FMT}'),
                                key=lambda q: q.stat().st_mtime)
                if not staged:
                    raise FileNotFoundError(f'{study} det{det} ch{ch}: no staged .{FIG_FMT} appeared')
                target = figure_path(
                    f'sbndaccess_{study}_{det_label.replace(" ", "").lower()}'
                    f'_{CHANNEL_TOKENS[ch]}_phase3')
                shutil.move(staged[-1], target)
            else:
                plt.show()
            made += 1

print(f'{made} phase-3 panels rendered for tiers {PHASE3_ENABLED_TIERS}')
if suppressed:
    print('data points withheld: ' + ', '.join(suppressed))
if skipped:
    print('not produced yet: ' + ', '.join(skipped))

# Step 4: conditional constraint overlays

The observed $E_\nu$ distribution is used to condition each partner variable,
and the conditioned prediction is drawn over the data with its shrunken
uncertainty band.

Read from the `*_indep_joint` runs, where the 11 hist1d detector systematics
are split into independent per-observable copies. The earlier shared-axis form
gave the constrained half an energy-binned detector shape, which inflated its
$\chi^2$ by 11 (δp) and 16 (p_μ) units and produced a spurious tension. The 90
per-event reweights are unaffected either way and carry most of the
cross-block correlation.

**The constraint uses the whole partner block**, both channels together, since
that is the information actually available. Each panel then shows one channel,
and the $\chi^2$ quoted on it is restricted to the bins drawn -- so the number
matches the picture rather than describing bins that are not shown.

In [ ]:
from scipy.stats import chi2 as chi2_dist
from mutools.profit import conditioned
from matplotlib.lines import Line2D
from mutools.plotting.save import saver

# stem, how many channels the joint config has, and which of them are the
# CONSTRAINED variable (E_nu always occupies the first two).
PHASE4_PAIRS = {
    r'$\delta p$':   dict(stem='p4_enu_dp_indep_joint',    nch=4, partner=(2, 3),
                          xlabel=r'Reconstructed $\delta p$ [MeV/c]'),
    r'$p_{\mu}$':    dict(stem='p4_enu_pmu_indep_joint',   nch=4, partner=(2, 3),
                          xlabel=r'Reconstructed $p_{\mu}$ [MeV/c]'),
    r'$E_{ccqe}$':   dict(stem='p4_enu_eccqe_indep_joint', nch=3, partner=(2,),
                          xlabel=r'Reconstructed $E_{ccqe}$ [GeV]'),
}
PHASE4_ALPHA = 1.0          # near-detector exposure scale; 1.0 = nominal

# Which block conditions which. The joint covariance is symmetric, so both
# directions come from the SAME runs -- no rebuild is needed to flip.
#   False : E_nu constrains the partner variable  (the request's Step 4)
#   True  : the partner variable constrains E_nu  (the reverse test)
# The reverse is a different question, not a cross-check: E_nu is the variable
# the oscillation analysis actually fits, so asking how well the TKI and lepton
# variables pin it down speaks to how much the interaction model is doing for
# the energy spectrum.
PHASE4_FLIP = False
PHASE4_ENU_XLABEL = r'Reconstructed Visible Energy [GeV]'
# Subchannel index -> (label, colour). The colour list in the plotting cells is
# written in XML DECLARATION order (intime, cos, nc, nuecc, othercc, target), so
# target takes the LAST colour, not the first. Pairing them the other way round
# inverts the whole stack relative to every other panel in this notebook.
P4_SUB = {0: ('In-time Cosmics', 'C0'), 1: ('Cosmics', 'C1'), 2: (r'$\nu$ NC', 'C2'),
          3: (r'$\nu_e$ CC', 'C4'), 4: (r'Other $\nu$ CC', 'C3'), 5: ('Target', 'C5')}
P4_ORANGE = '#d67a11'        # the colour mark_axis uses
P4_CHANNEL_NAME = {0: 'Single-Proton', 1: 'Multi-Proton'}


def _bins(hist, ch, prefix, sub):
    b = hist[(hist['variable'] == 0) & (hist['channel'] == ch) & (hist['detector'] == 0)
             & (hist['subchannel'] == sub) & (hist['prefix'] == prefix)]
    b = b.drop_duplicates('bin_index').sort_values('bin_index')
    width = (b['bin_high_edge'] - b['bin_low_edge']).to_numpy()
    return (b['bin_content'].to_numpy() * width,
            b['bin_low_edge'].to_numpy(), b['bin_high_edge'].to_numpy())


def phase4_constraint(spec):
    """Condition the partner block on E_nu; return everything a panel needs."""
    TOT = 4294967295
    hist = pd.DataFrame(uproot.open(REPO / f"{spec['stem']}_PROplot_histograms.root")['hist1d']
                        .arrays(library='np'))
    cv = [_bins(hist, c, 'CV', TOT)[0] for c in range(spec['nch'])]
    da = [_bins(hist, c, 'DATA', TOT)[0] for c in range(spec['nch'])]
    N, D = np.concatenate(cv), np.concatenate(da)
    C = uproot.open(REPO / f"{spec['stem']}_PROplot.root")['Covariance/collapsed_total_frac_cov'].values()
    nA = len(cv[0]) + len(cv[1])                       # E_nu occupies the first two channels
    enu, partner = np.arange(nA), np.arange(nA, len(N))
    # nd conditions, fd is conditioned
    nd, fd = (partner, enu) if PHASE4_FLIP else (enu, partner)
    S = np.diag(1.0 / np.maximum(N[nd], 1e-30))
    M = C[np.ix_(nd, nd)] + S / PHASE4_ALPHA
    shift = C[np.ix_(fd, nd)] @ np.linalg.solve(M, (D[nd] - N[nd]) / N[nd])
    resid = C[np.ix_(fd, fd)] - C[np.ix_(fd, nd)] @ np.linalg.solve(M, C[np.ix_(nd, fd)])
    return dict(hist=hist, N=N, D=D, C=C, A=nd, B=fd, nA=nA,
                mu_pre=N[fd], mu_post=N[fd] * (1 + shift),
                frac_pre=C[np.ix_(fd, fd)], frac_post=resid,
                reduction=conditioned(C, nd=nd, fd=fd, nd_stats=S, alpha=PHASE4_ALPHA).reduction)


def phase4_chi2(r, mu, frac, idx):
    """chi2 over a subset of the partner block -- the bins a panel actually draws."""
    d = r['D'][r['B']][idx]
    cov = frac[np.ix_(idx, idx)] * np.outer(mu[idx], mu[idx]) + np.diag(np.maximum(d, 1e-30))
    res = d - mu[idx]
    value = float(res @ np.linalg.solve(cov, res))
    return value, len(idx), float(chi2_dist.sf(value, len(idx)))

_dir = 'partner -> E_nu' if PHASE4_FLIP else 'E_nu -> partner'
print(f'phase 4: {len(PHASE4_PAIRS)} pairs, alpha = {PHASE4_ALPHA}, direction {_dir}')

One panel per channel per pair. The red-outlined band is the **conditioned**
uncertainty, the grey hatch the unconditioned one, and the dashed line the
conditioned prediction. The ratio panel is taken against the **unconditioned**
MC, so the conditioned prediction appears there as a departure from unity with
its own band.

Which block conditions which is set by `PHASE4_FLIP` in the configuration cell
above. The panels follow that toggle; the table two cells down reports both
directions at once.


In [ ]:
from mutools.plotting.profit import (add_error_band, construct_proxy_stack,
                                     construct_meta_handle, saver)
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

made = []
for label, spec in PHASE4_PAIRS.items():
    stem = spec['stem']
    if not (REPO / f'{stem}_PROplot_histograms.root').exists():
        print(f'{label}: MISSING {stem}'); continue
    r = phase4_constraint(spec)
    # channels of the CONSTRAINED block, and what to call the constrainer
    drawn = (0, 1) if PHASE4_FLIP else spec['partner']
    xlabel = PHASE4_ENU_XLABEL if PHASE4_FLIP else spec['xlabel']
    by = label if PHASE4_FLIP else r'$E_{\nu}^{reco}$'
    offset = 0
    for ch in drawn:
        content, lo, hi = _bins(r['hist'], ch, 'CV', 4294967295)
        n = len(content)
        idx = np.arange(offset, offset + n); offset += n
        ctr, wid, edges = 0.5 * (lo + hi), hi - lo, np.r_[lo, hi[-1]]
        mu_pre, mu_post = r['mu_pre'][idx], r['mu_post'][idx]
        sig_pre = np.sqrt(np.diag(r['frac_pre'])[idx]) * mu_pre
        sig_post = np.sqrt(np.clip(np.diag(r['frac_post'])[idx], 0, None)) * mu_post
        dat = r['D'][r['B']][idx]
        derr = np.sqrt(np.maximum(dat, 0))
        chi, ndf, pval = phase4_chi2(r, r['mu_post'], r['frac_post'], idx)

        fig = plt.figure(figsize=(7.2, 7.4))
        gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.02)
        ax = fig.add_subplot(gs[0, 0]); rx = fig.add_subplot(gs[1, 0], sharex=ax)

        # --- stack, in the declaration order mutools uses, with counter percentages
        sub_vals, sub_lab, sub_col = [], [], []
        for s in (5, 4, 3, 2, 1, 0):
            v = _bins(r['hist'], ch, 'CV', s)[0]
            if v.sum() <= 0: continue
            sub_vals.append(v); sub_lab.append(P4_SUB[s][0]); sub_col.append(P4_SUB[s][1])
        total = sum(v.sum() for v in sub_vals)
        labels = [f'{l} ({v.sum() / total:.1%})' for l, v in zip(sub_lab, sub_vals)]
        ax.hist([ctr] * len(sub_vals), bins=edges, weights=sub_vals, stacked=True,
                color=sub_col, edgecolor='none', alpha=0.7, rasterized=saver.rasterized)
        # Patches built directly rather than via construct_proxy_stack, which
        # assigns colours in REVERSED subchannel order -- that helper is correct
        # for mutools' own call site but mispairs labels when used standalone.
        proxy = [Patch(facecolor=c, alpha=0.7, label=l) for l, c in zip(labels, sub_col)]

        # --- unconstrained prior band, then the constrained prediction over it
        prior_patch = add_error_band(ax, edges, mu_pre, yerr=[sig_pre, sig_pre],
                                     label='Unconstrained Uncertainty',
                                     color='0.55', hatch='\\\\\\')
        # Drawn as a filled area with a red outline: the unconstrained band is
        # already grey hatch, and a second grey hatch on top of it is unreadable.
        ax.fill_between(np.r_[edges[0], ctr, edges[-1]],
                        np.r_[*(mu_post - sig_post)[[0]], mu_post - sig_post, *(mu_post - sig_post)[[-1]]],
                        np.r_[*(mu_post + sig_post)[[0]], mu_post + sig_post, *(mu_post + sig_post)[[-1]]],
                        step='mid', facecolor='red', alpha=0.12, zorder=3)
        for sgn in (-1, 1):
            y = mu_post + sgn * sig_post
            ax.step(np.r_[edges[0], ctr, edges[-1]], np.r_[y[0], y, y[-1]],
                    where='mid', color='red', lw=1.1, zorder=3)
        post_patch = Patch(facecolor='red', alpha=0.12, edgecolor='red',
                           label='Constrained Uncertainty')
        ax.step(np.r_[edges[0], ctr, edges[-1]], np.r_[mu_post[0], mu_post, mu_post[-1]],
                where='mid', color='k', ls='--', lw=1.5, zorder=4)
        post_line = Line2D([], [], color='k', ls='--', lw=1.5, label='Constrained MC')
        data_handle = ax.errorbar(ctr, dat, yerr=derr, xerr=0.5 * wid, fmt='ko', markersize=4,
                                  linewidth=1.0, capsize=2, zorder=5, label='Data')
        chi_patch = Patch(facecolor='none', edgecolor='none',
                          label=rf'$\chi^2$/ndf = {chi:.1f}/{ndf}')
        p_patch = Patch(facecolor='none', edgecolor='none', label=rf'$p$-value = {pval:.3f}')
        meta = construct_meta_handle(CODE_VERSION, SELECTION_VERSION)
        leg = ax.legend(handles=proxy + [prior_patch, post_patch, post_line, data_handle,
                                         chi_patch, p_patch, meta],
                        ncol=2, loc='upper right', fontsize=8,
                        title=f'{P4_CHANNEL_NAME.get(ch - drawn[0], "")}\n'
                              f'constrained by {by}', title_fontsize=11)
        leg.get_title().set_color(P4_ORANGE)
        leg.get_title().set_fontweight('bold')
        ax.set_ylabel('Events / bin'); ax.set_ylim(0, 1.75 * float(np.max(dat + derr))); ax.set_xlim(edges[0], edges[-1])
        ax.tick_params(labelbottom=False)
        mark_axis(ax, r'$\bf{SBND}$ Analysis-in-Progress', vadj=0.00)

        # --- ratio: everything relative to the UNCONSTRAINED MC, so the constrained
        #     prediction itself is visible as a departure from unity
        add_error_band(rx, edges, np.ones_like(mu_pre),
                       yerr=[sig_pre / mu_pre, sig_pre / mu_pre], label='',
                       color='0.55', hatch='\\\\\\')
        rlo, rhi = (mu_post - sig_post) / mu_pre, (mu_post + sig_post) / mu_pre
        rx.fill_between(np.r_[edges[0], ctr, edges[-1]], np.r_[rlo[0], rlo, rlo[-1]],
                        np.r_[rhi[0], rhi, rhi[-1]], step='mid',
                        facecolor='red', alpha=0.12, zorder=3)
        for y in (rlo, rhi):
            rx.step(np.r_[edges[0], ctr, edges[-1]], np.r_[y[0], y, y[-1]],
                    where='mid', color='red', lw=1.1, zorder=3)
        rx.step(np.r_[edges[0], ctr, edges[-1]],
                np.r_[(mu_post / mu_pre)[0], mu_post / mu_pre, (mu_post / mu_pre)[-1]],
                where='mid', color='k', ls='--', lw=1.5, zorder=4)
        rx.errorbar(ctr, dat / mu_pre, yerr=derr / mu_pre, xerr=0.5 * wid, fmt='ko',
                    markersize=4, linewidth=1.0, capsize=2, zorder=5)
        rx.axhline(1.0, color='red', linestyle='--', linewidth=1.0, zorder=1)
        rx.set_ylim(0.0, 2.0); rx.set_ylabel('Ratio to MC'); rx.set_xlabel(xlabel)
        if SAVE_TO:
            tok = CHANNEL_TOKENS.get(ch - drawn[0], f'ch{ch}')
            side = 'enu' if PHASE4_FLIP else 'partner'
            fig.savefig(figure_path(f'constrained_{stem}_{side}_{tok}'),
                        bbox_inches='tight')
        plt.show()
        made.append((label, ch - drawn[0], chi, ndf, pval,
                     float(np.mean(sig_pre / mu_pre)), float(np.mean(sig_post / mu_post))))

print(f'\n{len(made)} panels')
print(f"{'pair':14s}{'channel':>14}{'chi2/ndf':>12}{'p':>8}{'band pre':>10}{'band post':>11}")
for lab, ch, chi, ndf, pv, bpre, bpost in made:
    print(f'{lab:14s}{P4_CHANNEL_NAME.get(ch, ch):>14}{chi:8.1f}/{ndf:<3d}{pv:8.3f}{bpre:10.3f}{bpost:11.3f}')

### Both directions

The joint covariance is symmetric, so reversing the conditioning costs no new
runs -- only the index arrays swap. The two directions answer different
questions, though:

* **$E_\nu \rightarrow$ partner** is the request's Step 4: how much does the
  observed energy spectrum pin down the TKI and lepton variables.
* **partner $\rightarrow E_\nu$** is the reverse: how much do those variables
  pin down the spectrum the oscillation fit actually uses.

An internal check falls out of the reverse direction for free -- the
*unconditioned* $E_\nu$ $\chi^2$ must come out the same from all three joint
runs, since it is the same channel with the same systematics each time.

**Caveat, and it is a sharp one in the reverse direction.** The statistical
correlation between the two blocks is still neglected, and the two blocks are
filled by the *same events* -- the same selected interactions, binned two ways.
Conditioning therefore treats a fluctuation that is already present in the
constrained block as independent information about it: the shift is
over-applied and the residual covariance over-shrunk, both of which inflate the
post-constraint $\chi^2$. The forward direction has the same defect; it just
does not happen to bite there. Any reverse-direction $p$-value below 0.05
should be read as *not yet interpretable* rather than as a tension, until the
2D joint occupancy is available to supply the cross-block statistical term.

In [ ]:
# Both conditioning directions, from the same joint runs. Panels above follow
# PHASE4_FLIP; this table ignores it and reports both.
_P4_CH = {0: 'single-p', 1: 'multi-p', 2: 'single-p', 3: 'multi-p'}
_saved_flip = PHASE4_FLIP

rows = []
for _flip in (False, True):
    PHASE4_FLIP = _flip                       # phase4_constraint reads the global
    for _lab, _spec in PHASE4_PAIRS.items():
        if not (REPO / f"{_spec['stem']}_PROplot_histograms.root").exists():
            continue
        _r = phase4_constraint(_spec)
        _drawn, _off = ((0, 1) if _flip else _spec['partner']), 0
        for _ch in _drawn:
            _n = len(_bins(_r['hist'], _ch, 'CV', 4294967295)[0])
            _idx = np.arange(_off, _off + _n); _off += _n
            _pre = phase4_chi2(_r, _r['mu_pre'], _r['frac_pre'], _idx)
            _post = phase4_chi2(_r, _r['mu_post'], _r['frac_post'], _idx)
            _shift = np.abs(_r['mu_post'][_idx] / _r['mu_pre'][_idx] - 1).mean()
            rows.append((_flip, _lab, _P4_CH[_ch], _pre, _post, _shift,
                         float(np.mean(np.sqrt(np.clip(np.diag(_r['frac_pre'])[_idx], 0, None)))),
                         float(np.mean(np.sqrt(np.clip(np.diag(_r['frac_post'])[_idx], 0, None))))))

PHASE4_FLIP = _saved_flip
for _flip in (False, True):
    print('=' * 84)
    print('partner -> E_nu  (reverse)' if _flip else 'E_nu -> partner  (proposal Step 4)')
    print(f"{'pair':10s}{'channel':>10}{'chi2 pre':>13}{'p pre':>8}"
          f"{'chi2 post':>13}{'p post':>8}{'band pre':>10}{'band post':>10}{'|shift|':>9}")
    for f, lab, ch, pre, post, sh, bpre, bpost in rows:
        if f != _flip: continue
        print(f'{lab:10s}{ch:>10}{pre[0]:9.2f}/{pre[1]:<3d}{pre[2]:8.3f}'
              f'{post[0]:9.2f}/{post[1]:<3d}{post[2]:8.3f}{bpre:10.3f}{bpost:10.3f}{sh:9.3f}')

# The reverse direction reuses one E_nu channel three times, so its
# unconditioned chi2 must agree across pairs up to the allcov_seed noise floor.
for _ch in ('single-p', 'multi-p'):
    _v = [r[3][0] for r in rows if r[0] and r[2] == _ch]
    if len(_v) > 1:
        print(f'E_nu {_ch:8s} unconditioned chi2 across pairs: '
              f'{", ".join(f"{x:.2f}" for x in _v)}  (spread {max(_v) - min(_v):.2f})')

## Phase 5 -- Global Null Fit
Fit to null hypothesis (no oscillation) and look at pull results

In [ ]:
global_results = Path(REPO) / 'p5_vis_energy_null_null_global_fit.txt'
results = {'parameter': [], 'value': []}
with open(global_results) as f:
    lines = f.read().splitlines()
    for line in lines[1:]:
        if ':' not in line: continue
        name, value = line.split(':', 1)
        results['parameter'].append(name.strip())
        results['value'].append(float(value))

global_results = pd.DataFrame(results)

figure, ax = plt.subplots(figsize=(8, 6))
ax.hist(
    global_results['value'],
    bins=20,
    color='C0',
    edgecolor='k',
    alpha=0.7,
    range=(-2.0, 2.0)
)
ax.set_xlim(-2.0, 2.0)
ax.set_xlabel('Pull Strength')
ax.set_ylabel('Count')
mark_axis(ax, r'$\bf{SBND}$ Analysis-in-Progress')

if SAVE_TO:
    figure.savefig(figure_path('pulls_p5_vis_energy_null'), bbox_inches='tight')
plt.show()